# Stima dei costi per la generazione delle card (OpenRouter)

Inserisci nella cella **Parametri** il costo per milione di token in input e in output di ogni
modello (dalla pagina del modello su OpenRouter). Il notebook:

1. costruisce i **prompt reali** di generazione (`SYSTEM_PROMPT` + `build_card_prompt`) per i primi
   `TOP_K` chunk di ogni etichetta EDOS e ne conta i token di input;
2. stima i token di output per card (JSON visibile + eventuale ragionamento interno, che OpenRouter
   fattura come output);
3. calcola il costo: `input_tokens × prezzo_input + output_tokens × prezzo_output`.

I token sono contati con `tiktoken` (`o200k_base`, il tokenizer dei GPT recenti). Per i modelli con
tokenizer non pubblico (es. Claude) si applica un margine (`tokenizer_margin`).

In [25]:
import json
import sys
from pathlib import Path

import pandas as pd
import tiktoken

# Radice del progetto: la cartella che contiene src/unito_amazon.
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "src" / "unito_amazon").is_dir())
sys.path.insert(0, str(ROOT / "src"))

from unito_amazon.card_creation.prompts import SYSTEM_PROMPT, build_card_prompt
from unito_amazon.card_creation.schema import RankedPassage

pd.set_option("display.float_format", lambda value: f"{value:,.2f}")
print(ROOT)

c:\Users\micha\OneDrive\Desktop\Università\Progetti\GBV-CounterSpeech


## Parametri (da modificare)

- `MODELS`: prezzi in **$ per milione di token**. Un modello con prezzi `None` viene saltato.
- `OUTPUT_TOKENS_PER_CARD`: token del JSON di risposta (`reasoning` + `evidence`, 2-4 frasi ciascuno).
- `REASONING_TOKENS_PER_CARD`: token di ragionamento interno per card (0 se il ragionamento è disattivato).
- `RETRY_MARGIN`: margine per prove iniziali e ripetizioni dopo JSON non valido.
- `CONFIGURATIONS`: quando la coppia embedder + reranker migliore sarà scelta, lasciare solo quella.

In [30]:
MODELS = {
    # nome: {"input": $/M token, "output": $/M token, "tokenizer_margin": fattore sui token di input}
    "GPT-6.1 Sol (batch)":  {"input": 1.00, "output": 5.00, "tokenizer_margin": 1.3},
    "GPT-6.1 Terra (batch)":{"input": 0.50, "output": 2.50,  "tokenizer_margin": 1.3},
    "GPT-6.1 Luna (batch)": {"input": 0.05, "output": 0.25,  "tokenizer_margin": 1.3},
    "claude Sonnet 5.5 (batch)": {"input": 1, "output": 5,  "tokenizer_margin": 1.3},
    "claude Opus 5.5 (batch)": {"input": 2, "output": 10,  "tokenizer_margin": 1.3},
    "claude Haiku 5.5 (batch)": {"input": 0.05, "output": 0.25,  "tokenizer_margin": 1.3},
    "DeepSeek: DeepSeek V4.1 Flash 1017 (batch)": {"input": 0.011, "output": 0.20,  "tokenizer_margin": 1.3}
}

OUTPUT_TOKENS_PER_CARD = 300
REASONING_TOKENS_PER_CARD = 0
RETRY_MARGIN = 0.20

TOP_K = 15
RANKING_DIR = ROOT / "data/card_creation_comparison_edos_v2_words_clean_v4_20261104/work/reranking"
CONFIGURATIONS = [
    "bge-m3__bge-reranker-v2-m3",
    #"bge-m3__qwen3-reranker-8b",
    #"qwen3-embedding-8b__bge-reranker-v2-m3",
    #"qwen3-embedding-8b__qwen3-reranker-8b",
]

## Token di input (dai prompt reali)

In [31]:
encoding = tiktoken.get_encoding("o200k_base")
system_tokens = len(encoding.encode(SYSTEM_PROMPT))

records = []
for configuration in CONFIGURATIONS:
    path = RANKING_DIR / configuration / "edos_passages_ranked.jsonl"
    with path.open(encoding="utf-8") as handle:
        passages = [RankedPassage.model_validate(json.loads(line)) for line in handle]
    for passage in passages:
        if passage.reranker_rank <= TOP_K:
            records.append({
                "configuration": configuration,
                "label": passage.label,
                "reranker_rank": passage.reranker_rank,
                "input_tokens": system_tokens + len(encoding.encode(build_card_prompt(passage))),
            })
prompts = pd.DataFrame(records)

input_summary = prompts.groupby("configuration")["input_tokens"].agg(
    cards="count", total="sum", mean="mean", max="max")
input_summary

,cards,total,mean,max
configuration,,,,
bge-m3__bge-reranker-v2-m3,165,145626,882.58,1921


## Costo per modello e configurazione

In [32]:
rows = []
for configuration, summary in input_summary.iterrows():
    for model, price in MODELS.items():
        if price["input"] is None or price["output"] is None:
            continue
        input_tokens = summary["total"] * price["tokenizer_margin"]
        output_tokens = summary["cards"] * (OUTPUT_TOKENS_PER_CARD + REASONING_TOKENS_PER_CARD)
        input_cost = input_tokens / 1e6 * price["input"]
        output_cost = output_tokens / 1e6 * price["output"]
        rows.append({
            "configuration": configuration, "model": model,
            "input_tokens": input_tokens, "output_tokens": output_tokens,
            "input_cost_usd": input_cost, "output_cost_usd": output_cost,
            "cost_usd": input_cost + output_cost,
            "cost_usd_with_margin": (input_cost + output_cost) * (1 + RETRY_MARGIN),
        })
costs = pd.DataFrame(rows)

skipped = [model for model, price in MODELS.items() if price["input"] is None or price["output"] is None]
if skipped:
    print("Modelli saltati (prezzi mancanti):", skipped)
costs.pivot_table(index="model", columns="configuration", values="cost_usd_with_margin")

configuration,bge-m3__bge-reranker-v2-m3
model,
DeepSeek: DeepSeek V4.1 Flash 1017 (batch),0.01
GPT-6.1 Luna (batch),0.03
GPT-6.1 Sol (batch),0.52
GPT-6.1 Terra (batch),0.26
claude Haiku 5.5 (batch),0.03
claude Opus 5.5 (batch),1.05
claude Sonnet 5.5 (batch),0.52


In [29]:
# Totale per i modelli scelti, sulla configurazione più costosa (caso peggiore
# finché la coppia migliore non è scelta).
CHOSEN_MODELS = ["GPT-6.1 Sol (batch)", "GPT-6.1 Luna (batch)", "claude Sonnet 5.5 (batch)", "claude Haiku 5.5 (batch)"]

worst = input_summary["total"].idxmax()
chosen = costs[(costs.configuration == worst) & costs.model.isin(CHOSEN_MODELS)]
print(f"Configurazione: {worst}  ({int(input_summary.loc[worst, 'cards'])} card per modello)\n")
print(chosen[["model", "input_tokens", "output_tokens", "input_cost_usd", "output_cost_usd",
              "cost_usd_with_margin"]].to_string(index=False))
print(f"\nTotale ({len(chosen)} modelli con prezzi, margine incluso): {chosen.cost_usd_with_margin.sum():,.2f} $")

Configurazione: bge-m3__bge-reranker-v2-m3  (165 card per modello)

                    model  input_tokens  output_tokens  input_cost_usd  output_cost_usd  cost_usd_with_margin
      GPT-6.1 Sol (batch)    189,313.80      49,500.00            0.19             0.25                  0.52
     GPT-6.1 Luna (batch)    189,313.80      49,500.00            0.01             0.01                  0.03
claude Sonnet 5.5 (batch)    189,313.80      49,500.00            0.19             0.25                  0.52
 claude Haiku 5.5 (batch)    189,313.80      49,500.00            0.01             0.01                  0.03

Totale (4 modelli con prezzi, margine incluso): 1.10 $


## Note

- Dopo una prova reale su 2-3 card, sostituire `OUTPUT_TOKENS_PER_CARD` e `REASONING_TOKENS_PER_CARD`
  con i valori di `usage` restituiti da OpenRouter.
- I modelli di ragionamento fatturano i token interni come output: con ragionamento attivo
  `REASONING_TOKENS_PER_CARD` può valere diverse centinaia o migliaia di token per card.
- La stima non include il futuro LLM-as-a-judge.